In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
from model import MazeTransformer
import torch
import yaml

STATE_DICT_PATH = './models/smart_boi_short/model'
with open("./models/smart_boi_short/config.yaml", "r") as f:
    config = yaml.safe_load(f)

model = MazeTransformer(config)
model.load_state_dict(torch.load(STATE_DICT_PATH, weights_only=True))
model.eval()

MazeTransformer(
  (embedding_layer): Embedding(10, 256)
  (pos_embedding_layer): Embedding(64, 256)
  (transformer_blocks): ModuleList(
    (0-9): 10 x TransformerBlock(
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=256, out_features=768, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0, inplace=False)
      )
    )
  )
  (ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=5, bias=True)
)

In [47]:
from train import MazeDataset, create_causal_mask
import os

train_dataset = MazeDataset(os.path.join("./data", "train"))
sequences, targets, sizes = train_dataset[100]

sequences = sequences.unsqueeze(0)
targets = targets.unsqueeze(0)
sizes = sizes.unsqueeze(0)

seq_len = sequences.shape[1]
masks = create_causal_mask(sizes, seq_len)

sequences = sequences.to(config["device"])
masks = masks.to(config["device"])

model_out = model.forward(sequences, masks).squeeze()

sz = sizes.item()
predictions = model_out.squeeze()[sz - 1: sz - 1 + targets[0].shape[0]]

In [48]:
print("Predicted:", torch.argmax(predictions, dim=-1))
print("Target:   ", targets.squeeze())

Predicted: tensor([0, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 2, 0, 2, 2, 0, 0, 3, 3, 3, 4],
       device='mps:0')
Target:    tensor([0, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 2, 0, 2, 2, 0, 0, 3, 3, 3, 4])


In [56]:
ex_in = sequences[0][:sizes[0]].to('mps')
print(ex_in)
generated_path = model.generate(ex_in, max_tokens=100)
print(generated_path)

tensor([6, 6, 5, 6, 5, 6, 6, 6, 7, 5, 6, 5, 6, 6, 6, 5, 6, 7, 5, 6, 5, 6, 5, 5,
        6, 6, 7, 6, 6, 5, 6, 6, 5, 6, 5, 7, 5, 6, 6, 5, 5, 6, 6, 6, 7, 6, 5, 6,
        6, 5, 6, 5, 6, 7, 6, 5, 6, 5, 6, 6, 5, 6, 7, 6, 6, 6, 6, 6, 5, 6, 6, 7,
        8], device='mps:0')
tensor([0, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 2, 0, 2, 2, 0, 0, 3, 3, 3, 4],
       device='mps:0')


In [51]:
targets[0]

tensor([0, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 2, 0, 2, 2, 0, 0, 3, 3, 3, 4])

In [55]:
(generated_path == targets[0].to('mps')).all()

tensor(True, device='mps:0')